# Experiment 2 on Kaggle: CUB protocols B (fixed clusters) and R (refinement from GCD clusters)

**Inputs.** GPU or CPU; Internet on (the official CUB archive is streamed from data.caltech.edu and MD5-checked).
Attach the output of the Experiment 1 CUB notebook (it contains `e1_runs/cub`). These stages also run locally
(`README.md`); this notebook is for machines without the Experiment 1 outputs.

## 1. Code and environment

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

DATASET = "cub"
REPO, REF = "eylulpelinkilic/generalized-category-discovery", "main"   # set REF to a commit to pin
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
EXP1 = CODE / "Experiment1_GCD_CLIP_Baseline"
EXP2 = CODE / "Experiment2_SCD"
E1_RUNS = WORK / "e1_runs"
E2_OUT = WORK / "e2_outputs"
STATUS, T0 = {}, time.time()


def run(cmd, cwd=None, allow_fail=False):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or EXP2, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-12:]
    code = p.wait()
    if code and not allow_fail:
        raise RuntimeError(f"exit {code}: {cmd}\n" + "\n".join(tail))
    return code


def step(name, fn, needs=()):
    missing = [n for n in needs if STATUS.get(n) != "ok"]
    if missing:
        STATUS[name] = f"skipped: needs {missing}"
        return
    try:
        fn()
        STATUS[name] = "ok"
    except Exception as exc:
        STATUS[name] = f"failed: {exc}"
        print(f"!! {name} failed: {exc}")


if not CODE.exists():
    run(f"git clone -q https://github.com/{REPO}.git {CODE}", cwd=WORK)
run(f"git fetch -q origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit", COMMIT)
run("pip install -q open_clip_torch==3.3.0 ftfy regex nltk ortools", cwd=WORK)
run("pip install -q git+https://github.com/openai/CLIP.git", cwd=WORK)
import torch
print(torch.__version__, torch.cuda.is_available())

## 2. Restore previous outputs (if attached)

In [ ]:
def restore(name, target):
    for depth in range(5):
        for cand in Path("/kaggle/input").glob("*/" * depth + name):
            if not target.exists():
                print("restoring", cand, "->", target)
                shutil.copytree(cand, target)
            return str(cand)
    return None


PREV_E1 = restore(f"e1_runs/{DATASET}", E1_RUNS / DATASET)
PREV_E2 = restore("e2_outputs", E2_OUT)
print("previous Experiment 1 outputs:", PREV_E1, "| previous Experiment 2 outputs:", PREV_E2)

## 3. Data and commands

In [ ]:
assert (E1_RUNS / "cub" / "grounding").exists(), "attach the Experiment 1 CUB notebook output (e1_runs/cub)"
E2 = (f"python -m e2 --config configs/cub.yaml --set paths.outputs={E2_OUT} --set e1.runs_root={E1_RUNS} "
      f"--set dataset.archive_cache={WORK / 'cub_images'}")

## 4. Stages (each skips completed work)

In [ ]:
step("vocabulary", lambda: run(f"{E2} vocab"))
step("embeddings", lambda: run(f"{E2} cub-embed --clip vitb32"))
step("protocol_B", lambda: run(f"{E2} fixed --clip vitb32"), needs=["embeddings"])
for c in (0, 1, 2):
    for mode in ("unsupervised", "partially_supervised"):
        step(f"protocol_R_c{c}_{mode}", lambda c=c, mode=mode: run(f"{E2} refine-gcd --cluster-seed {c} --mode {mode}"),
             needs=["embeddings"])
step("report", lambda: run(f"{E2} report"))

## 5. Status and results archive

In [ ]:
(E2_OUT / f"NOTEBOOK_STATUS_{DATASET}.json").write_text(json.dumps({"commit": COMMIT, "status": STATUS}, indent=2))
zip_path = WORK / f"e2_results_{DATASET}.zip"
if zip_path.exists():
    zip_path.unlink()
# small download: results, traces, errors and galleries; features, assets, caches and top-k files stay in this
# version's output for the next session
run(f"cd {E2_OUT} && zip -qr {zip_path} . -x 'cache/*' 'assets/*' '*/features/*' '*/topk/*' '*/inputs/*/embeddings.npz'",
    cwd=WORK)
if (EXP2 / "reports").exists():
    shutil.copytree(EXP2 / "reports", WORK / "e2_reports", dirs_exist_ok=True)
print("=" * 60)
for k, v in STATUS.items():
    print(f"{k:24s} {v}")
print("=" * 60)